In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Procore — Commitment Change Orders
# MAGIC For every project in the company, pulls all Commitment Change Orders
# MAGIC and saves raw JSON to the Bronze lakehouse.

# COMMAND ----------

import requests
import json
import time
import datetime
from pyspark.sql import Row

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Auth — run the existing procore_auth notebook to get a token

# COMMAND ----------

auth_result = mssparkutils.notebook.run("procore_auth", 90)
auth_data = json.loads(auth_result)

ACCESS_TOKEN = auth_data["token"]
COMPANY_ID = auth_data["company_id"]

BASE_URL = "https://api.procore.com"

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Procore-Company-Id": str(COMPANY_ID),
}

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Look up all projects in the company

# COMMAND ----------

projects_resp = requests.get(
    f"{BASE_URL}/rest/v1.0/projects",
    headers=HEADERS,
    params={"company_id": COMPANY_ID, "serializer_view": "compact", "per_page": 200},
)
projects_resp.raise_for_status()
all_projects = projects_resp.json()
project_ids = [p["id"] for p in all_projects]
print(f"Found {len(project_ids)} project(s)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Helper: GET with pagination and 429 retry/backoff

# COMMAND ----------

def get_all_pages(url, params=None, per_page=100, max_retries=5):
    params = dict(params or {})
    params["per_page"] = per_page

    all_results = []
    page = 1
    while True:
        params["page"] = page

        retries = 0
        while True:
            resp = requests.get(url, headers=HEADERS, params=params)

            if resp.status_code == 429:
                retries += 1
                if retries > max_retries:
                    resp.raise_for_status()
                wait_seconds = int(resp.headers.get("Retry-After", 30))
                print(f"  -> 429 rate limited, waiting {wait_seconds}s (retry {retries}/{max_retries})...")
                time.sleep(wait_seconds)
                continue

            if not resp.ok:
                print(f"  -> {resp.status_code} on {resp.url}")
                print(f"  -> body: {resp.text[:500]}")
            resp.raise_for_status()
            break

        batch = resp.json()

        if not isinstance(batch, list):
            raise ValueError(f"Expected a list response, got: {type(batch)} -> {batch}")

        all_results.extend(batch)

        if len(batch) < per_page:
            break
        page += 1

    return all_results

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Helper: pull commitment change orders for one project

# COMMAND ----------

def get_commitment_change_orders_for_project(project_id):
    url = f"{BASE_URL}/rest/v1.0/projects/{project_id}/commitment_change_orders"
    try:
        change_orders = get_all_pages(url, params={"project_id": project_id})
    except requests.HTTPError as e:
        print(f"  [project {project_id}] skipped — {e}")
        return []

    for co in change_orders:
        co["_project_id"] = project_id

    return change_orders

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Loop over every project

# COMMAND ----------

all_change_orders = []

for project_id in project_ids:
    print(f"Project {project_id}...")
    change_orders = get_commitment_change_orders_for_project(project_id)
    print(f"  -> {len(change_orders)} commitment change order(s)")
    all_change_orders.extend(change_orders)

print(f"\nTotal commitment change orders pulled: {len(all_change_orders)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Write raw results to Bronze lakehouse

# COMMAND ----------

BRONZE_TABLE = "procore_commitment_change_orders"

pull_ts = datetime.datetime.utcnow().isoformat()

bronze_rows = [
    Row(
        project_id=co.get("_project_id"),
        change_order_id=co.get("id"),
        raw_json=json.dumps(co),
        pulled_at=pull_ts,
    )
    for co in all_change_orders
]

if bronze_rows:
    bronze_df = spark.createDataFrame(bronze_rows)
    bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
    print(f"Wrote {bronze_df.count()} row(s) to {BRONZE_TABLE}")
else:
    print("No commitment change orders pulled this run — nothing written to bronze.")

StatementMeta(, 430adf75-3aaa-4daf-b42d-fde46224de68, 3, Finished, Available, Finished, False)

Found 18 project(s)
Project 562949955001573...
  -> 36 commitment change order(s)
Project 562949954833574...
  -> 184 commitment change order(s)
Project 562949955118102...
  -> 3 commitment change order(s)
Project 562949955225798...
  -> 80 commitment change order(s)
Project 562949955257421...
  -> 0 commitment change order(s)
Project 562949955064640...
  -> 54 commitment change order(s)
Project 562949955318524...
  -> 0 commitment change order(s)
Project 562949955286476...
  -> 5 commitment change order(s)
Project 562949955375634...
  -> 1 commitment change order(s)
Project 562949954973730...
  -> 58 commitment change order(s)
Project 562949954973684...
  -> 1 commitment change order(s)
Project 562949954971827...
  -> 16 commitment change order(s)
Project 562949953807489...
  -> 1 commitment change order(s)
Project 562949955365267...
  -> 0 commitment change order(s)
Project 562949953807474...
  -> 0 commitment change order(s)
Project 562949954507004...
  -> 99 commitment change order